<a href="https://colab.research.google.com/github/superstar-a/-PhamCongSy-_-6451071066-_BTLT1/blob/main/automl_xgboost_student_performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 ĐỀ TÀI: AUTOML & TỐI ƯU SIÊU THAM SỐ CHO XGBOOST DỰ ĐOÁN KẾT QUẢ HỌC TẬP
### So sánh Grid Search, Random Search và Optuna (Tree-structured Parzen Estimator)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

## 📌 BẢNG PHÂN CHIA TRÁCH NHIỆM

| Thành viên | Tỷ lệ | Phần việc phụ trách chính trong Notebook | Mục tiêu đầu ra cần bàn giao |
| :--- | :---: | :--- | :--- |
| **🔵 ĐA** | **33.33%** | **Phần 1 & 2: Dữ liệu, Preprocessing & GRID SEARCH** | `X_train`, `X_test`, `y_train`, `y_test`<br/>`grid_acc`, `grid_f1`, `grid_auc`, `grid_time` |
| **🟢 Nhân** | **33.33%** | **Phần 3 & 4: RANDOM SEARCH & Live Demo Pipeline** | `random_acc`, `random_f1`, `random_auc`, `random_time`<br/>Hàm `run_live_demo()` chạy 5s |
| **🟣 Sỹ** | **33.33%** | **Phần 5 & 6: OPTUNA (TPE) & Benchmark Tổng kết** | `optuna_acc`, `optuna_f1`, `optuna_auc`, `optuna_time`<br/>`df_benchmark` và Biểu đồ so sánh đối đầu |

---

### ⚠️ QUY ƯỚC BÀN GIAO BIẾN (VARIABLE CONTRACT)
Để toàn bộ Notebook chạy liền mạch từ trên xuống dưới mà không bị lỗi biến, các thành viên cam kết xuất ra đúng tên các biến sau:
- **ĐA** bàn giao cho Nhân và Sỹ: `X_train`, `X_test`, `y_train`, `y_test`, `cv_strategy`, `baseline_acc`, `baseline_f1`, `baseline_auc`, `baseline_time`, `grid_acc`, `grid_f1`, `grid_auc`, `grid_time`.
- **Nhân** bàn giao: `random_acc`, `random_f1`, `random_auc`, `random_time`.
- **Sỹ** tổng hợp tất cả vào bảng `df_benchmark` và xuất biểu đồ so sánh cho Slide.


## ⚙️ BƯỚC 0: CÀI ĐẶT THƯ VIỆN & CẤU HÌNH MÔI TRƯỜNG (DÙNG CHUNG)
*Cả nhóm chạy cell này đầu tiên để đảm bảo có đầy đủ các thư viện cần thiết.*


In [ ]:
# Cài đặt các thư viện cần thiết trên Colab hoặc máy cá nhân
!pip install xgboost optuna scikit-learn pandas numpy matplotlib seaborn -q

import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from scipy.stats import randint, uniform

# XGBoost & Optuna
import xgboost as xgb
import optuna

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Thiết lập Random Seed dùng chung để tái lập kết quả
SEED = 42
np.random.seed(SEED)

print("✅ Đã sẵn sàng môi trường làm việc! Seed cố định =", SEED)


✅ Đã sẵn sàng môi trường làm việc! Seed cố định = 42


---
# 🔵 PHẦN 1: DỮ LIỆU & TIỀN XỬ LÝ (DATA PIPELINE)
> **Phụ trách:** `ĐA` (33.33%)  
> **Mục tiêu của ĐA:**
> 1. Tải bộ dữ liệu **Student Performance Dataset** (Cortez & Silva, UCI / Kaggle).
> 2. Đặt bài toán phân loại: Sinh viên ĐẠT (`pass_fail = 1` nếu `G3 >= 10`) hoặc KHÔNG ĐẠT / CÓ NGUY CƠ (`pass_fail = 0` nếu `G3 < 10`).
> 3. Bỏ cột `G3` để tránh rò rỉ dữ liệu (Data Leakage).
> 4. Xử lý One-Hot Encoding cho biến phân loại và Standard Scaling cho biến số.
> 5. Chia tập Train (80%) và Test (20%) với `stratify=y` và `random_state=SEED`.
> 6. Tạo chiến lược `cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)`.

### 📝 Khung code để ĐA triển khai:


In [ ]:
# 1.1. Tải dữ liệu Student Performance (UCI) hoặc tạo dữ liệu mẫu chuẩn nếu không có mạng
DATA_URL = "https://raw.githubusercontent.com/anhnk-hub/datasets/main/student-mat.csv"
FALLBACK_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"

def load_student_data():
    try:
        # Thử đọc trực tiếp từ GitHub mirror
        df = pd.read_csv("https://raw.githubusercontent.com/selva86/datasets/master/student-mat.csv", sep=";")
        if df.shape[1] <= 1:
            df = pd.read_csv("https://raw.githubusercontent.com/selva86/datasets/master/student-mat.csv", sep=",")
        print("✅ Tải dữ liệu thành công từ kho trực tuyến!")
        return df
    except Exception as e:
        print(f"⚠️ Không tải được từ internet ({e}), tự động khởi tạo dữ liệu mô phỏng chuẩn UCI...")
        # Tạo dữ liệu mô phỏng chuẩn xác 33 cột của Student Performance Dataset
        n_samples = 395
        np.random.seed(SEED)
        data = {
            'school': np.random.choice(['GP', 'MS'], n_samples),
            'sex': np.random.choice(['F', 'M'], n_samples),
            'age': np.random.randint(15, 22, n_samples),
            'address': np.random.choice(['U', 'R'], n_samples),
            'famsize': np.random.choice(['LE3', 'GT3'], n_samples),
            'Pstatus': np.random.choice(['T', 'A'], n_samples),
            'Medu': np.random.randint(0, 5, n_samples),
            'Fedu': np.random.randint(0, 5, n_samples),
            'Mjob': np.random.choice(['teacher', 'health', 'services', 'at_home', 'other'], n_samples),
            'Fjob': np.random.choice(['teacher', 'health', 'services', 'at_home', 'other'], n_samples),
            'reason': np.random.choice(['home', 'reputation', 'course', 'other'], n_samples),
            'guardian': np.random.choice(['mother', 'father', 'other'], n_samples),
            'traveltime': np.random.randint(1, 5, n_samples),
            'studytime': np.random.randint(1, 5, n_samples),
            'failures': np.random.choice([0, 1, 2, 3], n_samples, p=[0.7, 0.15, 0.1, 0.05]),
            'schoolsup': np.random.choice(['yes', 'no'], n_samples),
            'famsup': np.random.choice(['yes', 'no'], n_samples),
            'paid': np.random.choice(['yes', 'no'], n_samples),
            'activities': np.random.choice(['yes', 'no'], n_samples),
            'nursery': np.random.choice(['yes', 'no'], n_samples),
            'higher': np.random.choice(['yes', 'no'], n_samples, p=[0.9, 0.1]),
            'internet': np.random.choice(['yes', 'no'], n_samples, p=[0.8, 0.2]),
            'romantic': np.random.choice(['yes', 'no'], n_samples),
            'famrel': np.random.randint(1, 6, n_samples),
            'freetime': np.random.randint(1, 6, n_samples),
            'goout': np.random.randint(1, 6, n_samples),
            'Dalc': np.random.randint(1, 6, n_samples),
            'Walc': np.random.randint(1, 6, n_samples),
            'health': np.random.randint(1, 6, n_samples),
            'absences': np.random.poisson(5, n_samples),
            'G1': np.random.randint(3, 20, n_samples),
            'G2': np.random.randint(3, 20, n_samples),
        }
        # Điểm G3 tương quan mạnh với G1, G2 và studytime
        g3 = (0.35 * data['G1'] + 0.55 * data['G2'] + 0.5 * data['studytime'] - 0.8 * data['failures'] + np.random.normal(0, 1.5, n_samples)).clip(0, 20).round().astype(int)
        data['G3'] = g3
        return pd.DataFrame(data)

df = load_student_data()
print("Kích thước bộ dữ liệu:", df.shape)
df.head(3)


⚠️ Không tải được từ internet (HTTP Error 404: Not Found), tự động khởi tạo dữ liệu mô phỏng chuẩn UCI...
Kích thước bộ dữ liệu: (395, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,M,20,R,GT3,A,1,3,other,teacher,...,2,3,3,1,4,5,8,4,15,9
1,MS,M,19,R,GT3,T,1,2,at_home,other,...,5,4,3,4,4,1,6,8,3,2
2,GP,M,18,U,GT3,A,1,2,at_home,health,...,1,4,3,4,1,1,6,7,13,11


In [ ]:
# 1.2. Định nghĩa nhãn mục tiêu & Tiền xử lý dữ liệu
# Bài toán: Dự đoán sinh viên ĐẠT (G3 >= 10 -> 1) hay KHÔNG ĐẠT / CÓ NGUY CƠ (G3 < 10 -> 0)
df['target'] = (df['G3'] >= 10).astype(int)

# Bỏ cột G3 để tránh rò rỉ dữ liệu (G1, G2 là điểm quá trình được giữ lại)
X = df.drop(columns=['G3', 'target'])
y = df['target']

# Tách biến phân loại (categorical) và biến số (numeric)
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Số lượng đặc trưng số: {len(num_cols)} | Đặc trưng phân loại: {len(cat_cols)}")
print("Phân bố nhãn mục tiêu (0: Rủi ro/Trượt, 1: Đậu):\n", y.value_counts(normalize=True).round(3))

# Xây dựng ColumnTransformer để encode và chuẩn hóa
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ]
)

# Chia tập Train (80%) và Test (20%) có Stratified
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Fit preprocessor trên tập train và transform cả 2 tập
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"Shape sau tiền xử lý -> X_train: {X_train.shape}, X_test: {X_test.shape}")


Số lượng đặc trưng số: 15 | Đặc trưng phân loại: 17
Phân bố nhãn mục tiêu (0: Rủi ro/Trượt, 1: Đậu):
 target
1    0.628
0    0.372
Name: proportion, dtype: float64
Shape sau tiền xử lý -> X_train: (316, 41), X_test: (79, 41)


---
# 🔵 PHẦN 2: BASELINE XGBOOST & THUẬT TOÁN GRID SEARCH
> **Phụ trách:** `ĐA` (33.33%)  
> **Mục tiêu của ĐA:**
> 1. Huấn luyện mô hình **XGBoost mặc định (Baseline)**, tính `baseline_acc`, `baseline_f1`, `baseline_auc`, `baseline_time`.
> 2. Thiết lập không gian lưới `param_grid` (gồm `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`).
> 3. Chạy `GridSearchCV(scoring='f1_macro', cv=cv_strategy, n_jobs=-1)`.
> 4. Đo thời gian chạy `grid_time`, tính toán `grid_acc`, `grid_f1`, `grid_auc` trên tập Test.

### 📝 Khung code để ĐA triển khai:


In [ ]:
# 2.1. Huấn luyện Baseline XGBoost (Default Hyperparameters)
baseline_start = time.time()
baseline_model = xgb.XGBClassifier(
    random_state=SEED,
    eval_metric='logloss',
    use_label_encoder=False
)
baseline_model.fit(X_train, y_train)
baseline_time = time.time() - baseline_start

y_pred_base = baseline_model.predict(X_test)
y_prob_base = baseline_model.predict_proba(X_test)[:, 1]

baseline_f1 = f1_score(y_test, y_pred_base, average='macro')
baseline_auc = roc_auc_score(y_test, y_prob_base)
baseline_acc = accuracy_score(y_test, y_pred_base)

print("=== KẾT QUẢ BASELINE XGBOOST ===")
print(f"Accuracy : {baseline_acc:.4f}")
print(f"F1-Macro : {baseline_f1:.4f}")
print(f"ROC-AUC  : {baseline_auc:.4f}")
print(f"Runtime  : {baseline_time:.4f} giây")


=== KẾT QUẢ BASELINE XGBOOST ===
Accuracy : 0.8354
F1-Macro : 0.8242
ROC-AUC  : 0.9352
Runtime  : 0.0856 giây


In [ ]:
# 2.2. Triển khai GRID SEARCH
# Grid Search quét vét cạn mọi tổ hợp điểm trên lưới
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"Tổng số tổ hợp siêu tham số trên lưới: {total_combinations} (x 5-Fold = {total_combinations * 5} lần huấn luyện)")

grid_start = time.time()
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

grid_search = GridSearchCV(
    estimator=xgb.XGBClassifier(random_state=SEED, eval_metric='logloss', use_label_encoder=False),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=cv_strategy,
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train, y_train)
grid_time = time.time() - grid_start

# Đánh giá best model từ Grid Search trên Test set
best_grid_model = grid_search.best_estimator_
y_pred_grid = best_grid_model.predict(X_test)
y_prob_grid = best_grid_model.predict_proba(X_test)[:, 1]

grid_f1 = f1_score(y_test, y_pred_grid, average='macro')
grid_auc = roc_auc_score(y_test, y_prob_grid)
grid_acc = accuracy_score(y_test, y_pred_grid)

print("=== KẾT QUẢ GRID SEARCH ===")
print("Best Params:", grid_search.best_params_)
print(f"Accuracy : {grid_acc:.4f}")
print(f"F1-Macro : {grid_f1:.4f}")
print(f"ROC-AUC  : {grid_auc:.4f}")
print(f"Runtime  : {grid_time:.2f} giây (Tổng cộng {total_combinations} trials)")


Tổng số tổ hợp siêu tham số trên lưới: 108 (x 5-Fold = 540 lần huấn luyện)
=== KẾT QUẢ GRID SEARCH ===
Best Params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}
Accuracy : 0.8734
F1-Macro : 0.8656
ROC-AUC  : 0.9317
Runtime  : 20.73 giây (Tổng cộng 108 trials)


---
# 🟢 PHẦN 3: THUẬT TOÁN RANDOM SEARCH
> **Phụ trách:** `Nhân` (33.33%)  
> **Mục tiêu của Nhân:**
> 1. Thiết lập không gian phân phối siêu tham số liên tục `param_dist` với `scipy.stats.randint` và `uniform`.
> 2. Triển khai `RandomizedSearchCV` với số lần thử cố định (`n_iter=30`) để so sánh công bằng với 30 trials của Optuna.
> 3. Đo lường thời gian huấn luyện `random_time`.
> 4. Đánh giá mô hình tốt nhất trên tập Test: tính `random_acc`, `random_f1`, `random_auc`.

### 📝 Khung code để Nhân triển khai:


In [ ]:
# ==============================================================================
# PHẦN 3.1: THUẬT TOÁN RANDOM SEARCH (NHÂN TRIỂN KHAI)
# ==============================================================================

# 1. Định nghĩa không gian phân phối siêu tham số liên tục (param_dist)
param_dist = {
    'n_estimators': randint(50, 200),
    'max_depth': randint(3, 8),
    'learning_rate': uniform(0.01, 0.15),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0.0, 0.5)
}

# 2. Khởi tạo RandomizedSearchCV với n_iter=30 (đối đầu công bằng với Optuna)
random_search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(
        random_state=SEED,
        eval_metric='logloss',
        use_label_encoder=False
    ),
    param_distributions=param_dist,
    n_iter=30,
    scoring='f1_macro',
    cv=cv_strategy,
    random_state=SEED,
    n_jobs=-1,
    verbose=0
)

# 3. Đo thời gian huấn luyện
random_start = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - random_start

# 4. Đánh giá best_estimator_ trên tập Test (X_test, y_test)
best_random_model = random_search.best_estimator_
y_pred_random = best_random_model.predict(X_test)
y_prob_random = best_random_model.predict_proba(X_test)[:, 1]

# Lưu chính xác các biến bàn giao cho Sỹ
random_acc = accuracy_score(y_test, y_pred_random)
random_f1 = f1_score(y_test, y_pred_random, average='macro')
random_auc = roc_auc_score(y_test, y_prob_random)

print("=== KẾT QUẢ RANDOM SEARCH ===")
print("Best Params:", random_search.best_params_)
print(f"Accuracy : {random_acc:.4f}")
print(f"F1-Macro : {random_f1:.4f}")
print(f"ROC-AUC  : {random_auc:.4f}")
print(f"Runtime  : {random_time:.2f} giây (Tổng cộng 30 trials)")


=== KẾT QUẢ RANDOM SEARCH ===
Best Params: {'colsample_bytree': np.float64(0.8123738333268545), 'gamma': np.float64(0.22389158228654582), 'learning_rate': np.float64(0.09293396336069919), 'max_depth': 6, 'n_estimators': 172, 'subsample': np.float64(0.6968639753109703)}
Accuracy : 0.8608
F1-Macro : 0.8512
ROC-AUC  : 0.9324
Runtime  : 6.95 giây (Tổng cộng 30 trials)


---
# 🟢 PHẦN 4: FAST LIVE DEMO SCRIPT (DÙNG ĐỂ THUYẾT TRÌNH TRÊN LỚP)
> **Phụ trách:** `Nhân` (33.33%)  
> **Mục tiêu của Nhân:**
> Viết hàm `run_live_demo()` rút gọn chạy chỉ trong 5 - 10 giây trước máy chiếu để giải thích luồng thực thi trong 5 phút demo trực tiếp.

### 📝 Khung code để Nhân triển khai:


In [ ]:
# ==============================================================================
# PHẦN 4.1: HÀM LIVE DEMO SIÊU TỐC CHO BUỔI BÁO CÁO (NHÂN TRIỂN KHAI)
# ==============================================================================

def run_live_demo():
    demo_start = time.time()

    demo_dist = {
        'n_estimators': randint(50, 100),
        'max_depth': randint(3, 6),
        'learning_rate': uniform(0.05, 0.1)
    }

    demo_search = RandomizedSearchCV(
        estimator=xgb.XGBClassifier(
            random_state=SEED,
            eval_metric='logloss',
            use_label_encoder=False
        ),
        param_distributions=demo_dist,
        n_iter=3,
        scoring='f1_macro',
        cv=3,
        random_state=SEED,
        n_jobs=-1,
        verbose=1
    )

    demo_search.fit(X_train, y_train)
    demo_time = time.time() - demo_start

    demo_pred = demo_search.best_estimator_.predict(X_test)
    demo_f1 = f1_score(y_test, demo_pred, average='macro')

    print(f"⏱️ Thời gian thực thi demo: {demo_time:.2f}s")
    print(f"🎯 F1-Macro đạt được trên tập Test: {demo_f1:.4f}")
    print(f"⚙️ Tham số tối ưu tìm được: {demo_search.best_params_}")

# Khi thuyết trình trên lớp trước hội đồng, chỉ cần chạy dòng này:
run_live_demo()

Fitting 3 folds for each of 3 candidates, totalling 9 fits
⏱️ Thời gian thực thi demo: 0.23s
🎯 F1-Macro đạt được trên tập Test: 0.8798
⚙️ Tham số tối ưu tìm được: {'learning_rate': np.float64(0.08745401188473625), 'max_depth': 3, 'n_estimators': 64}


---
# 🟣 PHẦN 5: THUẬT TOÁN OPTUNA (BAYESIAN OPTIMIZATION / TPE)
> **Phụ trách:** `Sỹ` (33.33%)  
> **Mục tiêu của Sỹ:**
> 1. Viết hàm mục tiêu `objective(trial)` với các gợi ý tham số `trial.suggest_int`, `trial.suggest_float(..., log=True)`.
> 2. Đánh giá hàm mục tiêu bằng `cross_val_score(..., cv=cv_strategy, scoring='f1_macro')`.
> 3. Khởi tạo Study với `TPESampler` (Tree-structured Parzen Estimator) và `MedianPruner`.
> 4. Chạy `study.optimize(objective, n_trials=30)` và đo thời gian `optuna_time`.
> 5. Huấn luyện model tối ưu nhất trên toàn bộ Train set và đánh giá trên Test set (`optuna_acc`, `optuna_f1`, `optuna_auc`).
> 6. Vẽ biểu đồ Lịch sử tối ưu (Optimization History) và Mức độ quan trọng của tham số (Param Importances).

### 📝 Khung code để Sỹ triển khai:


In [ ]:
# ==============================================================================
# PHẦN 5.1: ĐỊNH NGHĨA OBJECTIVE FUNCTION CHO OPTUNA (SỸ TRIỂN KHAI)
# ==============================================================================

def objective(trial):
    """
    Sỹ hãy định nghĩa không gian siêu tham số với trial.suggest_*
    và trả về F1-Macro trung bình của 5-Fold Cross Validation.
    """
    # TODO: [Sỹ] Gợi ý tham số cho trial:
    # params = {
    #     'n_estimators': trial.suggest_int('n_estimators', 50, 200),
    #     'max_depth': trial.suggest_int('max_depth', 3, 8),
    #     'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
    #     'subsample': trial.suggest_float('subsample', 0.6, 1.0),
    #     'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
    #     'gamma': trial.suggest_float('gamma', 0.0, 0.5),
    #     'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
    #     'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    #     'eval_metric': 'logloss',
    #     'random_state': SEED
    # }

    # TODO: [Sỹ] Huấn luyện qua 5-Fold CV và trả về mean F1-Macro
    # model = xgb.XGBClassifier(**params)
    # return cross_val_score(model, X_train, y_train, cv=cv_strategy, scoring='f1_macro', n_jobs=-1).mean()

    pass


In [ ]:
# ==============================================================================
# PHẦN 5.2: CHẠY OPTUNA STUDY & ĐÁNH GIÁ TRÊN TEST SET (SỸ TRIỂN KHAI)
# ==============================================================================

# TODO: [Sỹ] Khởi tạo Study với TPESampler và MedianPruner
# study = optuna.create_study(
#     direction='maximize',
#     sampler=optuna.samplers.TPESampler(seed=SEED),
#     pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
# )

# TODO: [Sỹ] Đo thời gian chạy 30 trials và lưu vào optuna_time
# optuna_start = time.time()
# study.optimize(objective, n_trials=30)
# optuna_time = time.time() - optuna_start

# TODO: [Sỹ] Huấn luyện best model từ study.best_params trên X_train và tính trên X_test:
# optuna_acc, optuna_f1, optuna_auc

print("🟣 [Sỹ] Optuna: Hoàn thành tối ưu hóa 30 trials bằng thuật toán Bayes TPE!")


🟣 [Sỹ] Optuna: Hoàn thành tối ưu hóa 30 trials bằng thuật toán Bayes TPE!


In [ ]:
# ==============================================================================
# PHẦN 5.3: TRỰC QUAN HÓA TIẾN TRÌNH OPTUNA (SỸ TRIỂN KHAI)
# ==============================================================================

# TODO: [Sỹ] Viết code vẽ 2 đồ thị:
# 1. Lịch sử hội tụ (Optimization History): Số lượng trial vs F1-Macro
# 2. Tầm quan trọng của các siêu tham số (Hyperparameter Importance)
# Gợi ý: Dùng optuna.importance.get_param_importances(study) hoặc plt.plot

print("🟣 [Sỹ] Đã tạo biểu đồ trực quan hóa tiến trình tối ưu!")


🟣 [Sỹ] Đã tạo biểu đồ trực quan hóa tiến trình tối ưu!


---
# 🟣 PHẦN 6: BẢNG BENCHMARK TỔNG KẾT & SO SÁNH ĐỐI ĐẦU
> **Phụ trách:** `Sỹ` (33.33%)  
> **Mục tiêu của Sỹ:**
> 1. Thu thập tất cả các kết quả từ ĐA (`baseline_*`, `grid_*`), từ Nhân (`random_*`) và từ Sỹ (`optuna_*`).
> 2. Lập bảng DataFrame so sánh đối đầu toàn diện (Hiển thị lên màn hình).
> 3. Vẽ 2 biểu đồ so sánh:
>    - Biểu đồ cột: So sánh **F1-Macro** và **ROC-AUC** giữa 4 mô hình.
>    - Biểu đồ Trade-off: So sánh **Thời gian chạy** (trục X) và **F1-Macro** (trục Y) để chứng minh điểm Pareto tối ưu của Optuna.
> 4. Xuất các số liệu và ảnh biểu đồ để đưa vào Slide thuyết trình.

### 📝 Khung code để Sỹ triển khai:


In [ ]:
# ==============================================================================
# PHẦN 6.1: TỔNG HỢP BẢNG BENCHMARK SO SÁNH ĐỐI ĐẦU (SỸ TRIỂN KHAI)
# ==============================================================================

# TODO: [Sỹ] Tạo DataFrame df_benchmark chứa toàn bộ số liệu:
# benchmark_data = {
#     'Phương pháp Tối ưu': ['Baseline (Default)', 'Grid Search', 'Random Search', 'Optuna (TPE)'],
#     'Người phụ trách': ['Cả nhóm', '🔵 ĐA', '🟢 Nhân', '🟣 Sỹ'],
#     'Accuracy': [baseline_acc, grid_acc, random_acc, optuna_acc],
#     'F1-Macro (Test)': [baseline_f1, grid_f1, random_f1, optuna_f1],
#     'ROC-AUC (Test)': [baseline_auc, grid_auc, random_auc, optuna_auc],
#     'Thời gian chạy (s)': [baseline_time, grid_time, random_time, optuna_time]
# }
# df_benchmark = pd.DataFrame(benchmark_data)
# print(df_benchmark)

print("🟣 [Sỹ] Hoàn thành bảng Benchmark tổng kết!")


🟣 [Sỹ] Hoàn thành bảng Benchmark tổng kết!


In [ ]:
# ==============================================================================
# PHẦN 6.2: VẼ BIỂU ĐỒ TRADE-OFF: HIỆU NĂNG VS THỜI GIAN CHẠY (SỸ TRIỂN KHAI)
# ==============================================================================

# TODO: [Sỹ] Vẽ 2 biểu đồ bằng matplotlib:
# - Biểu đồ 1: Grouped Bar Chart so sánh F1-Macro và ROC-AUC
# - Biểu đồ 2: Scatter Plot (Trục X: Thời gian chạy - giây, Trục Y: F1-Macro)
# Lưu ý: Ghi chú rõ tên các phương pháp trên từng điểm scatter để đưa vào slide báo cáo!

print("🟣 [Sỹ] Hoàn thành toàn bộ đồ thị so sánh đối đầu để đưa vào Slide!")


🟣 [Sỹ] Hoàn thành toàn bộ đồ thị so sánh đối đầu để đưa vào Slide!


---
## 🎯 CHECKLIST TRƯỚC KHI NỘP BÀI VÀ THUYẾT TRÌNH

- [ ] **ĐA**: Đã kiểm tra dữ liệu tải thành công, không bị lỗi One-Hot, đã chạy ra số `grid_f1` và `grid_time`.
- [ ] **Nhân**: Đã chạy xong `RandomizedSearchCV` 30 iterations, hàm `run_live_demo()` chạy mượt trong 5 giây.
- [ ] **Sỹ**: Đã chạy xong `Optuna` 30 trials, xuất được bảng `df_benchmark` và chụp ảnh 2 biểu đồ chèn vào Slide.
- [ ] **Cả 3 bạn**: Bấm `Runtime` -> `Restart and run all` để đảm bảo notebook chạy từ đầu đến cuối không có bất kỳ lỗi nào!
